# 02 · Cleaning the data and creating new features

**Goal:** turn the unreadable codes (`A11`, `A43`, ...) into readable text, and create a few new columns that help the analysis.
At the end we save a clean copy of the data for the next notebooks.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/german_credit.csv")
df.head(3)

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1


## 1. Translating the codes
A **dictionary** (`{key: value}`) is the perfect tool for a translation table: the code is the key, and the meaning is the value.

In [2]:
# For every coded column: a dictionary {code: meaning}, taken from DATA_DICTIONARY.md
code_meanings = {
 "checking_status": {"A11": "< 0 DM", "A12": "0-200 DM", "A13": ">= 200 DM", "A14": "no account"},
 "credit_history": {"A30": "no credits / all paid", "A31": "all paid at this bank", "A32": "paid on time so far", "A33": "delays in the past", "A34": "critical account"},
 "purpose": {"A40": "new car", "A41": "used car", "A42": "furniture/equipment", "A43": "radio/TV", "A44": "appliances", "A45": "repairs", "A46": "education", "A47": "vacation", "A48": "retraining", "A49": "business", "A410": "other"},
 "savings": {"A61": "< 100 DM", "A62": "100-500 DM", "A63": "500-1000 DM", "A64": ">= 1000 DM", "A65": "unknown / none"},
 "employment": {"A71": "unemployed", "A72": "< 1 year", "A73": "1-4 years", "A74": "4-7 years", "A75": ">= 7 years"},
 "personal_status": {"A91": "male, divorced/separated", "A92": "female, divorced/separated/married", "A93": "male, single", "A94": "male, married/widowed", "A95": "female, single"},
 "other_debtors": {"A101": "none", "A102": "co-applicant", "A103": "guarantor"},
 "property": {"A121": "real estate", "A122": "savings agreement / life insurance", "A123": "car or other", "A124": "unknown / none"},
 "other_installment_plans": {"A141": "bank", "A142": "stores", "A143": "none"},
 "housing": {"A151": "rent", "A152": "own", "A153": "for free"},
 "job": {"A171": "unemployed / unskilled non-resident", "A172": "unskilled resident", "A173": "skilled employee", "A174": "management / highly qualified"},
 "telephone": {"A191": "none", "A192": "yes"},
 "foreign_worker": {"A201": "yes", "A202": "no"},
}


`.map(dictionary)` replaces every value in a column with its partner from the dictionary.
For example, `df["housing"].map({"A151": "rent", "A152": "own", "A153": "for free"})` turns every `A152` into `"own"`.

The loop below does that for all 13 coded columns at once.

In [3]:
# Go through every coded column and replace the codes with their meaning
for column, meanings in code_meanings.items():
    df[column] = df[column].map(meanings)

df.head(3)

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,class
0,< 0 DM,6,critical account,radio/TV,1169,unknown / none,>= 7 years,4,"male, single",none,...,real estate,67,none,own,2,skilled employee,1,yes,yes,1
1,0-200 DM,48,paid on time so far,radio/TV,5951,< 100 DM,1-4 years,2,"female, divorced/separated/married",none,...,real estate,22,none,own,1,skilled employee,1,none,yes,2
2,no account,12,critical account,education,2096,< 100 DM,4-7 years,2,"male, single",none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,1


## 2. Checking that nothing went wrong
If a code was missing from our dictionary, `.map()` would leave an **empty value (NaN)** in its place.
So we count the empty values again. It should still be 0 everywhere.

In [4]:
# Total number of empty cells in the whole table (sum per column, then sum of those)
df.isnull().sum().sum()

np.int64(0)

**What I see:** 0 missing values, so every code was translated.

## 3. Creating new columns (feature engineering)
New columns made from existing ones often reveal patterns more clearly.

| New column | How it's made | Why it's useful |
|---|---|---|
| `is_bad` | 1 if `class` is 2, else 0 | Easier to read than 1/2, and its **mean = the bad rate** |
| `monthly_payment` | loan amount ÷ months | How heavy the loan is each month |
| `age_group` | age cut into 4 groups | Lets us compare age groups |
| `long_loan` | True if longer than 24 months | Lets us compare short and long loans |

In [5]:
# class == 2 gives True/False; .astype(int) turns True -> 1, False -> 0
df["is_bad"] = (df["class"] == 2).astype(int)

# Vectorized division: one line for all 1,000 customers, no loop
df["monthly_payment"] = (df["credit_amount"] / df["duration"]).round(1)

# pd.cut puts each age into a bin: (18, 25], (25, 35], (35, 50], (50, 100]
df["age_group"] = pd.cut(df["age"], bins=[18, 25, 35, 50, 100], labels=["19-25", "26-35", "36-50", "51+"])

# A comparison gives True/False for every customer
df["long_loan"] = df["duration"] > 24

df[["credit_amount", "duration", "monthly_payment", "age", "age_group", "long_loan", "class", "is_bad"]].head()

,credit_amount,duration,monthly_payment,age,age_group,long_loan,class,is_bad
0,1169,6,194.8,67,51+,False,1,0
1,5951,48,124.0,22,19-25,True,2,1
2,2096,12,174.7,49,36-50,False,1,0
3,7882,42,187.7,45,36-50,True,1,0
4,4870,24,202.9,53,51+,False,2,1


## 4. Saving the clean data
`to_csv` writes the table to a file. `index=False` means "don't save the row numbers as an extra column".

In [6]:
df.to_csv("data/german_credit_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (1000, 25)


## Summary of this notebook
1. All 13 coded columns were **translated into readable text** with dictionaries and `.map()`, with no values lost.
2. **4 new features** were created: `is_bad`, `monthly_payment`, `age_group` and `long_loan`.
3. The clean data is saved as `data/german_credit_clean.csv` and used in `03_analysis.ipynb`.